#### Environment Check


In [ ]:
import sys
print(sys.executable)


#### Setup


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field


#### Project Paths


In [ ]:
cwd = Path.cwd()

if (cwd / "code").exists() and (cwd / "data").exists():
    LAB_DIR = cwd
elif (cwd.parent / "code").exists() and (cwd.parent / "data").exists():
    LAB_DIR = cwd.parent
else:
    LAB_DIR = Path("..").resolve()

CODE_DIR = LAB_DIR / "code"
DATA_DIR = LAB_DIR / "data"
REPORTS_DIR = LAB_DIR / "reports"

REPORTS_DIR.mkdir(exist_ok=True)

str(LAB_DIR)


#### Import Course Helpers


In [ ]:
import sys

sys.path.append(str(CODE_DIR))

from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress


#### Load OpenAI Client


In [ ]:
PROJECT_ROOT = LAB_DIR.parents[1]
ENV_PATH = PROJECT_ROOT / ".env"
loaded = load_dotenv(ENV_PATH)
print("Env file:", ENV_PATH)
print("Env loaded:", loaded)
openai_client = OpenAI()
MODEL = "gpt-5.4-mini"


#### Load RAG Answers


In [ ]:
rags_path = DATA_DIR / "rag-answers-new.csv"

df_answers = pd.read_csv(rags_path)
answers = df_answers.to_dict(orient="records")

len(answers)


#### Define Judge Output


In [ ]:
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )


#### Judge Instructions


In [ ]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()


#### Test Judge Prompt


In [ ]:
rec = answers[0]

prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"],
)

print(prompt)


#### Judge One Answer


In [ ]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
    model=MODEL,
)

eval_result


#### Check One Judge Cost


In [ ]:
calc_price(usage)


#### Create Judge Function


In [ ]:
def evaluate_aqa(question, answer_orig, answer_llm, model=MODEL):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm,
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage


#### Create Record Judge Function


In [ ]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"],
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_orig": rec["answer_orig"],
        "answer_llm": rec["answer_llm"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage


#### Run LLM Judge


In [ ]:
answers_to_judge = answers
# answers_to_judge = answers[:20]

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers_to_judge, judge_record)


#### Split Evaluations And Usage


In [ ]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

len(evaluations)


#### Create Evaluation DataFrame


In [ ]:
df_eval = pd.DataFrame(evaluations)

df_eval.head()


#### Check Judge Scores


In [ ]:
score_counts = df_eval["score"].value_counts()
score_rate = df_eval["score"].value_counts(normalize=True)

score_counts, score_rate


#### Calculate Judge Cost


In [ ]:
total_cost = calc_total_price(usages)

total_cost


#### Inspect Bad Answers


In [ ]:
df_eval[df_eval["score"] == "bad"].head()


#### Save Judge Results


In [ ]:
output_path = DATA_DIR / "rag-evaluations-new.csv"
df_eval.to_csv(output_path, index=False)

output_path


#### Save Judge Report


In [ ]:
good_count = int((df_eval["score"] == "good").sum())
bad_count = int((df_eval["score"] == "bad").sum())
total_count = len(df_eval)

good_rate = good_count / total_count

report_path = REPORTS_DIR / "rag_judge_metrics.md"

report_lines = [
    "# RAG LLM-as-a-Judge Metrics",
    "",
    f"- Total answers judged: {total_count}",
    f"- Good answers: {good_count}",
    f"- Bad answers: {bad_count}",
    f"- Good rate: {good_rate:.2%}",
    f"- Judge cost: {total_cost}",
    f"- Output file: {output_path.name}",
]

with open(report_path, "w") as f:
    f.write("\n".join(report_lines))
    f.write("\n")

report_path
